# M31 · Locating an earthquake

**Week 4 · about 10.5 h (+ 1 h optional)** · before this: M20, M30

> **한국어 요약** 도달 시각으로 진원과 발생 시각을 구하는 격자 탐색과 가이거 방법을 직접 만들고, 위치의 오차를 계산합니다. 직접 고른 피킹으로 2019년 상주 지진(ML 3.9)의 진원을 구해 기상청 목록과 비교하고, 2018년 작은 지진의 깊이가 얼마나 잘 정해지는지 살펴봅니다. 마지막으로 같은 피킹으로 HYPOINVERSE를 실행합니다.

**You will**

- Locate a practice earthquake by grid search and Geiger's method
- Compute the errors of a location and test them by simulation
- Locate the 2019 Sangju mainshock and compare it with KMA
- Test how well a small earthquake's depth is known; run HYPOINVERSE

**New words:** absolute location (절대 진원 결정), grid search (격자 탐색), Geiger's method (가이거 방법), Jacobian matrix (야코비 행렬), covariance matrix (공분산 행렬), error correlation (오차 상관 (상관 계수)), depth scan (깊이 탐색 (고정 깊이 탐색)), HYPOINVERSE (진원 결정 프로그램) · [all terms](../../../docs/glossary.md)

**Optional reading:** Havskov, Bormann & Schweitzer (2012), sections 1-4; Husen & Hardebeck (2010), what makes a location accurate

Log any AI help in `work/ai_log/M31.md` ([rules](../../../docs/ai_policy.md)); pass criteria: [criteria/M31.yaml](../../../.sgtl/criteria/M31.yaml).

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Run it first, then the setup cell.
DATASET = "sangju2019"                    # your STP download (M03) in the data folder
KMA_EVENT_ID = "2019009762"               # the ML 3.9 Sangju mainshock, 21 July 2019 (KMA id)
SMALL_EVENT_ID = "2018000078"             # the ML 1.4 Sangju earthquake, 25 Feb 2018 KST (KMA id)
PICKS_CSV = "../picks/M20_manual_picks.csv"           # your manual picks of M20
SMALL_PICKS_CSV = "../picks/M31_2018_manual_picks.csv"  # your 2018 picks (Part 6 writes them)
MODEL_NAME = "kim1983"                    # velocity model of Parts 5-7
MAX_DISTANCE_KM = 100.0                   # use picks within this distance of KMA's epicenter
GRID_HALF_KM = 20.0                       # grid search: +- this many km east and north
GRID_STEP_KM = 1.0                        # grid search: node spacing (km)
GRID_DEPTHS_KM = (0.5, 30.0, 1.0)         # grid search: first depth, last depth, step (km)
SCAN_DEPTHS_KM = (0.5, 25.0, 0.5)         # depth scan (task 5): first, last, step (km)
SMALL_BAND = "HH"                         # Part 6: the band you pick on
SMALL_MAX_KM = 60.0                       # Part 6: the picker lists stations within this distance
TRIAL_DEPTH_KM = 10.0                     # task 6: HYPOINVERSE's trial depth (lab setting)
CHANGED_TRIAL_DEPTH_KM = 5.0              # task 6: the one setting you change
PRACTICE_SEED = 31                        # seed of the practice earthquake (Part 1)
FIGURE_DIR = "figures"                    # your figures go here
RESULT_DIR = "results"                    # your result files go here

In [ ]:
import sgtl_tutorial as st
S = st.init("M31")

import filecmp
import os
import shutil
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import UTCDateTime, read
from pyproj import Proj
from scipy.interpolate import RectBivariateSpline

Path(FIGURE_DIR).mkdir(exist_ok=True)
Path(RESULT_DIR).mkdir(exist_ok=True)

## Part 1 · Four unknowns from arrival times (about 25 min)

**Absolute location** finds an earthquake's hypocenter, where the rupture starts ($x$ east, $y$ north, depth $z$; in
km), and its origin time $t_0$. Your data are the arrival times of P and S waves: your picks.

The cell makes a practice earthquake: ten stations, picks with random errors of 0.05 s. It prints the true
hypocenter, so you can check your locators on it.

In [ ]:
V_HALF = {"P": 6.0, "S": 6.0 / 1.73}         # km/s: P and S speed of the practice half-space


def tt_half(distance_km, depth_km, phase):
    """Travel time (s) in a homogeneous half-space: straight rays at V_HALF[phase] km/s."""
    return np.sqrt(np.asarray(distance_km, dtype=float) ** 2 + float(depth_km) ** 2) / V_HALF[phase]


practice_rng = np.random.default_rng(PRACTICE_SEED)
PRACTICE_TRUTH = {"x_km": 3.4, "y_km": -1.8, "z_km": 8.6, "t0_s": 0.7}
angles = np.sort(practice_rng.uniform(0.0, 2.0 * np.pi, 10))
ranges = practice_rng.uniform(6.0, 45.0, 10)
practice_stations = pd.DataFrame({"station": [f"XX.P{k:02d}" for k in range(1, 11)],
                                  "x_km": np.round(ranges * np.sin(angles), 2),
                                  "y_km": np.round(ranges * np.cos(angles), 2)})
rows = []
for _, sta in practice_stations.iterrows():
    for phase in ("P", "S"):
        if phase == "S" and sta["station"] in ("XX.P03", "XX.P08"):
            continue                                  # two stations without an S pick
        dist = np.hypot(sta["x_km"] - PRACTICE_TRUTH["x_km"], sta["y_km"] - PRACTICE_TRUTH["y_km"])
        travel = tt_half(np.array([dist]), PRACTICE_TRUTH["z_km"], phase)[0]
        rows.append({"station": sta["station"], "phase": phase, "x_km": sta["x_km"], "y_km": sta["y_km"],
                     "t_s": round(PRACTICE_TRUTH["t0_s"] + travel + practice_rng.normal(0.0, 0.05), 3)})
practice = pd.DataFrame(rows)
print("practice earthquake (made up): hypocenter", PRACTICE_TRUTH)
print(practice.to_string(index=False))

fig, ax = plt.subplots(figsize=(4.8, 4.6))
ax.plot(practice_stations["x_km"], practice_stations["y_km"], "^", color="tab:blue", ms=8, label="Stations")
for _, sta in practice_stations.iterrows():
    ax.annotate(sta["station"][3:], (sta["x_km"], sta["y_km"]), xytext=(4, 3), textcoords="offset points",
                fontsize=8)
ax.plot(PRACTICE_TRUTH["x_km"], PRACTICE_TRUTH["y_km"], "*", color="gold", mec="k", ms=14, label="True epicenter")
ax.set_xlabel("East (km)")
ax.set_ylabel("North (km)")
ax.set_aspect("equal")
ax.set_title("Practice network", fontsize=10)
fig.legend(loc="upper center", ncol=2, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "practice_network.png", dpi=150)

Each pick $i$ obeys

$$t_i = t_0 + T_i(x, y, z) + e_i$$

$T_i$ is the travel time that a velocity model predicts, and $e_i$ the error. The residual $r_i = t_i - t_0 - T_i$
is what the model misses. The RMS residual sums them up in one number:

$$\mathrm{RMS} = \sqrt{\frac{1}{N}\sum_{i=1}^{N} r_i^2}$$

Locating means finding the $x, y, z, t_0$ with the smallest RMS.

**The travel-time function.** Your locators get a function `tt(distance_km, depth_km, phase)`: the travel times for
a numpy array of epicentral distances from a source at one depth. Here it is `tt_half`: straight rays, with one P
and one S speed everywhere. From Part 5 on, your M30 functions build it.

## Part 2 · Grid search (about 20 min, then task 1)

A **grid search** computes the RMS at every node of a grid of trial hypocenters and keeps the best node. It needs no
starting point and cannot get stuck in a wrong dip. The price: many travel times, and an answer only as fine as the
grid.

At a node, $x, y, z$ are fixed, but $t_0$ is not. Write the residuals for $t_0 = 0$ as $\rho_i = t_i - T_i$; then
$r_i = \rho_i - t_0$. Which $t_0$ makes $\sum_i r_i^2$ smallest? Then $t_0$ needs no grid axis of its own.

### M31-T1 · A grid search over the hypocenter · core · about 75 min

Write `grid_search(obs, tt, xs, ys, zs)`. It returns the RMS at every node, each with its best origin time, and the
best node. The docstring lists every key.

Example: if every predicted travel time at a node is 1.5 s shorter than the observed time, the best origin time
there is 1.5 s and the RMS is 0.

Not allowed: `scipy`, `sklearn`, `statsmodels`, `torch`, `jax`, `lmfit`, `pyrocko` or the course package.

In [ ]:
# Before you code: the practice picks carry random errors of 0.05 s. What RMS do you expect at the true hypocenter?
st.predict("M31-T1", """

""")

In [ ]:
def grid_search(obs, tt, xs, ys, zs):
    """RMS residual at every node of a grid, and the best node.

    obs        : table with the columns phase ("P" or "S"), x_km, y_km (the station, km east and north) and t_s
                 (arrival time, s after a reference moment); any index and row order; do not change it
    tt         : function tt(distance_km, depth_km, phase) -> travel times (s) for a numpy array of epicentral
                 distances (km, along the surface); call it with each pick's phase
    xs, ys, zs : 1-D arrays of trial positions east, north and down (km)
    returns    : dict with
        rms     numpy array of shape (len(xs), len(ys), len(zs)); rms[i, j, k] is the RMS of all picks (P and S
                together) at the node (xs[i], ys[j], zs[k]), with the origin time that makes it smallest
        t0      numpy array of the same shape: that origin time at each node (s after the same reference moment)
        x_km, y_km, z_km, t0_s, rms_s   floats: the node with the smallest RMS, its origin time and its RMS
    """
    # Step 1: plain arrays from the table: station east and north, phase, arrival time.
    # Step 2: the epicentral distance from every (x, y) node to every pick's station, in one array.
    # Step 3: for each depth and phase, one call of tt with all those distances: the predicted travel times.
    # Step 4: at each node, the residuals for t0 = 0 are rho = t - T. Choose the t0 that makes
    #         sum (rho - t0)^2 smallest (where is the derivative with respect to t0 zero?), then the RMS.
    # Step 5: the node with the smallest RMS of the whole cube.
    # YOUR CODE HERE
    raise NotImplementedError("M31-T1: write your code here")

In [ ]:
st.check("M31-T1", func=grid_search)

Your grid search on the practice earthquake. The maps show the RMS around the best node.

In [ ]:
PRACTICE_X = np.arange(-15.0, 15.01, 1.0)
PRACTICE_Y = np.arange(-15.0, 15.01, 1.0)
PRACTICE_Z = np.arange(0.5, 20.01, 1.0)
start_clock = time.perf_counter()
practice_grid = grid_search(practice, tt_half, PRACTICE_X, PRACTICE_Y, PRACTICE_Z)
print(f"{practice_grid['rms'].size} nodes in {1000.0 * (time.perf_counter() - start_clock):.0f} ms")
print(f"best node: x {practice_grid['x_km']:g}, y {practice_grid['y_km']:g}, z {practice_grid['z_km']:g} km, "
      f"origin time {practice_grid['t0_s']:.3f} s, RMS {practice_grid['rms_s']:.3f} s")
print("true hypocenter:", PRACTICE_TRUTH)
i_best = int(np.argmin(np.abs(PRACTICE_X - practice_grid["x_km"])))
j_best = int(np.argmin(np.abs(PRACTICE_Y - practice_grid["y_km"])))
k_best = int(np.argmin(np.abs(PRACTICE_Z - practice_grid["z_km"])))
close = practice_grid["rms"] <= practice_grid["rms_s"] + 0.05      # nodes within 0.05 s of the best RMS
for label, inside, axis in (("east-west", close[:, j_best, k_best], PRACTICE_X),
                            ("north-south", close[i_best, :, k_best], PRACTICE_Y),
                            ("depth", close[i_best, j_best, :], PRACTICE_Z)):
    print(f"through the best node, {label}: RMS within 0.05 s of the best from {axis[inside].min():g} to "
          f"{axis[inside].max():g} km")

fig, (ax_m, ax_d) = plt.subplots(1, 2, figsize=(10.0, 4.2), width_ratios=[1.0, 1.25])
levels = np.linspace(0.0, 0.6, 25)
mapped = ax_m.contourf(PRACTICE_X, PRACTICE_Y, practice_grid["rms"][:, :, k_best].T, levels=levels,
                       cmap="viridis", extend="max")
ax_m.plot(practice_stations["x_km"], practice_stations["y_km"], "^", color="w", mec="k", ms=7)
ax_m.plot(practice_grid["x_km"], practice_grid["y_km"], "o", color="tab:red", mec="w", ms=7, label="Best node")
ax_m.plot(PRACTICE_TRUTH["x_km"], PRACTICE_TRUTH["y_km"], "*", color="gold", mec="k", ms=13, label="Truth")
ax_m.set_xlim(PRACTICE_X[0], PRACTICE_X[-1])
ax_m.set_ylim(PRACTICE_Y[0], PRACTICE_Y[-1])
ax_m.set_aspect("equal")
ax_m.set_xlabel("East (km)")
ax_m.set_ylabel("North (km)")
ax_m.set_title(f"(a) Map view at {PRACTICE_Z[k_best]:g} km depth", fontsize=10, loc="left")
ax_d.contourf(PRACTICE_X, PRACTICE_Z, practice_grid["rms"][:, j_best, :].T, levels=levels, cmap="viridis",
              extend="max")
ax_d.plot(practice_grid["x_km"], practice_grid["z_km"], "o", color="tab:red", mec="w", ms=7)
ax_d.plot(PRACTICE_TRUTH["x_km"], PRACTICE_TRUTH["z_km"], "*", color="gold", mec="k", ms=13)
ax_d.set_ylim(PRACTICE_Z[-1], PRACTICE_Z[0])
ax_d.set_xlabel("East (km)")
ax_d.set_ylabel("Depth (km)")
ax_d.set_title(f"(b) East-west section at {PRACTICE_Y[j_best]:g} km north", fontsize=10, loc="left")
fig.colorbar(mapped, ax=ax_d, label="RMS residual (s)")
fig.legend(loc="upper center", ncol=2, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "practice_grid_search.png", dpi=150)

**Explain:** Through the best node, in which direction does the RMS stay close to its best over the longest distance
(printed above), and why?

## Part 3 · Geiger's method (about 40 min, then task 2)

**Geiger's method** improves a location step by step. Near a trial hypocenter, a small change
$\delta m = (\delta x, \delta y, \delta z, \delta t_0)$ changes the predicted times almost linearly:

$$G\,\delta m \approx r$$

$G$ is the **Jacobian**: one row per pick, and the columns $\partial T_i/\partial x$, $\partial T_i/\partial y$,
$\partial T_i/\partial z$ (s/km) and 1 (for $t_0$). Get the derivatives by central differences: move the source by
$h = 0.01$ km each way, and divide the change of the time by $2h$.

The least-squares step solves a 4 x 4 system (`np.linalg.solve`):

$$(G^{\mathsf T} G + \lambda^2 I)\,\delta m = G^{\mathsf T} r$$

- **Damping** $\lambda^2$ keeps steps short where the data barely fix a direction.
- Repeat with new $r$ and $G$ until the steps become tiny.
- If a step would put the source above the ground (an **airquake**), halve the depth instead.

### M31-T2 · Geiger's method · core · about 120 min

Write `geiger(obs, tt, start, damping=0.01, max_iter=30, tol_km=0.001)`. It repeats the step above from `start`. It
returns the final location with its residuals, its Jacobian and whether it converged; the docstring lists every key.

Example: from the true hypocenter with exact picks, the first step is 0, so `n_iter` is 1 and `converged` is True.

Not allowed: `scipy`, `sklearn`, `statsmodels`, `torch`, `jax`, `tensorflow`, `lmfit`, `sympy`, `mpmath`, `pyrocko`
or the course package.

In [ ]:
# Before you code: how many steps will geiger need from the best node of your grid search, and from a start
# about 19 km away?
st.predict("M31-T2", """

""")

In [ ]:
def geiger(obs, tt, start, damping=0.01, max_iter=30, tol_km=0.001):
    """Geiger's method: damped least-squares steps from start until the hypocenter stops moving.

    obs      : table with phase, x_km, y_km, t_s (as in task 1); do not change it
    tt       : the travel-time function of task 1
    start    : (x_km, y_km, z_km, t0_s) to start from
    damping  : lambda in (G^T G + lambda^2 I) dm = G^T r
    max_iter : largest number of steps
    tol_km   : stop, converged, after a step that moves the hypocenter by less than this (km)
    Each step adds dm to x, y, z and t0, except that the new depth is z / 2 when z + dz < 0.
    returns  : dict with
        x_km, y_km, z_km, t0_s   floats after the last step
        rms_s       the RMS residual there
        residual_s  numpy array of the residuals there, one per row of obs, in its order
        G           numpy array (len(obs), 4), the Jacobian there: dT/dx, dT/dy, dT/dz (central differences,
                    h = 0.01 km) and a column of ones
        n_iter      int, the number of steps taken;  converged  bool
    """
    # Step 1: plain arrays from the table, and a small function that returns the predicted travel time of every
    #         pick for a trial (x, y, z) (as in task 1, for one point).
    # Step 2: the Jacobian: column 1 = (T(x + h) - T(x - h)) / (2 h), the same for y and z, column 4 = ones.
    # Step 3: one step: r = t - t0 - T; solve (G^T G + damping^2 I) dm = G^T r; update; apply the depth rule.
    # Step 4: repeat until the step in space is shorter than tol_km or max_iter steps are done; then the
    #         residuals, the RMS and the Jacobian at the final hypocenter.
    # YOUR CODE HERE
    raise NotImplementedError("M31-T2: write your code here")

In [ ]:
st.check("M31-T2", func=geiger)

Your `geiger` from the best node and from two poor starts 19 km away, at the surface and 5 km deep. The tables follow
the poor starts step by step. Their last column is the largest $|\partial T/\partial z|$ in `G`.

In [ ]:
from_grid = geiger(practice, tt_half, (practice_grid["x_km"], practice_grid["y_km"], practice_grid["z_km"],
                                       practice_grid["t0_s"]))
print(f"from the grid's best node: {from_grid['n_iter']} steps, converged {from_grid['converged']}: x "
      f"{from_grid['x_km']:.3f}, y {from_grid['y_km']:.3f}, z {from_grid['z_km']:.3f} km, origin time "
      f"{from_grid['t0_s']:.3f} s, RMS {from_grid['rms_s']:.4f} s")
miss = np.sqrt((from_grid["x_km"] - PRACTICE_TRUTH["x_km"]) ** 2 + (from_grid["y_km"] - PRACTICE_TRUTH["y_km"]) ** 2
               + (from_grid["z_km"] - PRACTICE_TRUTH["z_km"]) ** 2)
print(f"distance from the true hypocenter: {miss:.3f} km")
for label, poor_start in (("start at the surface", (-12.0, 9.0, 0.0, 0.0)),
                          ("start 5 km deep", (-12.0, 9.0, 5.0, 0.0))):
    path = []
    for k in range(1, 11):
        fit = geiger(practice, tt_half, poor_start, max_iter=k)
        path.append({"steps": fit["n_iter"], "x_km": fit["x_km"], "y_km": fit["y_km"], "z_km": fit["z_km"],
                     "t0_s": fit["t0_s"], "rms_s": fit["rms_s"], "converged": fit["converged"],
                     "max |dT/dz| (s/km)": float(np.abs(np.asarray(fit["G"])[:, 2]).max())})
        if fit["converged"]:
            break
    print(f"\n{label} {poor_start}:")
    print(pd.DataFrame(path).round(4).to_string(index=False))

**Explain:** From the surface start, why did the depth never leave 0 km, and what does `converged` not tell you?

## Part 4 · How well is the location known? (about 30 min, then task 3)

If every pick error is random, with the same standard deviation $\sigma$, the four unknowns have the **covariance
matrix**

$$C = \sigma^2 \left(G^{\mathsf T} G\right)^{-1}$$

with $G$ the Jacobian at the solution. The square roots of its diagonal are the standard errors of $x, y, z$ (km)
and $t_0$ (s): how much each would scatter if the picks were made again.

An off-diagonal term divided by the two standard errors is an **error correlation** (-1 to 1). Near ±1, the data
cannot tell the two unknowns apart: a trade-off.

### M31-T3 · Errors of a location · core · about 60 min

Write `location_errors(G, sigma_s)`. It returns the covariance matrix, the four standard errors and the correlation
of the depth and origin-time errors. The check also relocates a made-up earthquake 300 times with your `geiger`.

Example: `G = np.eye(4)` and `sigma_s = 2` give `cov` = 4 times the identity, standard errors of 2 and a correlation
of 0.

Not allowed: `scipy`, `statsmodels`, `sklearn`, `lmfit`, `numpy.polyfit`, `numpy.ma.polyfit`, `numpy.polynomial`,
`xarray`, `torch`, `jax` or the course package.

In [ ]:
# Before you code: which of x, y and z will have the largest standard error for the practice earthquake?
st.predict("M31-T3", """

""")

In [ ]:
def location_errors(G, sigma_s):
    """Covariance matrix, standard errors and depth-origin time correlation of a location.

    G       : Jacobian at the location, numpy array (N, 4) as geiger returns it: columns x, y, z (s/km) and the
              origin time (ones)
    sigma_s : standard deviation of one pick error (s), the same for every pick
    returns : dict with
        cov      4 x 4 numpy array, sigma^2 (G^T G)^-1 (km^2, km s and s^2)
        se       numpy array of the four standard errors: x, y, z in km, origin time in s
        corr_zt  float, the correlation coefficient of the depth and origin-time errors
    """
    # Step 1: C = sigma^2 (G^T G)^-1.
    # Step 2: the standard errors are the square roots of the diagonal of C.
    # Step 3: the correlation of depth (index 2) and origin time (index 3): C[2, 3] / (se_z se_t0).
    # YOUR CODE HERE
    raise NotImplementedError("M31-T3: write your code here")

In [ ]:
st.check("M31-T3", func=location_errors, locate=geiger)

Your errors of the practice location, and a **Monte Carlo** test as in the check: 200 relocations by your `geiger`,
each with new random pick errors of 0.05 s.

In [ ]:
practice_err = location_errors(from_grid["G"], 0.05)
print("your standard errors for 0.05 s pick errors: x {:.3f}, y {:.3f}, z {:.3f} km, origin time {:.3f} s; "
      "depth-origin time correlation {:.2f}".format(*practice_err["se"], practice_err["corr_zt"]))
mc_rng = np.random.default_rng(PRACTICE_SEED + 1)
exact = practice.copy()
exact["t_s"] = practice["t_s"].to_numpy() - from_grid["residual_s"]        # times that fit your location exactly
cloud = []
for _ in range(200):
    noisy = exact.copy()
    noisy["t_s"] = exact["t_s"].to_numpy() + mc_rng.normal(0.0, 0.05, len(exact))
    fit = geiger(noisy, tt_half, (from_grid["x_km"], from_grid["y_km"], from_grid["z_km"], from_grid["t0_s"]))
    cloud.append([fit["x_km"], fit["y_km"], fit["z_km"], fit["t0_s"]])
cloud = np.array(cloud)
print("scatter of 200 relocations:      x {:.3f}, y {:.3f}, z {:.3f} km, origin time {:.3f} s; "
      "correlation {:.2f}".format(*cloud.std(axis=0), np.corrcoef(cloud[:, 2], cloud[:, 3])[0, 1]))

values, vectors = np.linalg.eigh(practice_err["cov"][:2, :2])
angle = np.linspace(0.0, 2.0 * np.pi, 200)
ellipse = vectors @ (np.sqrt(values)[:, None] * np.vstack([np.cos(angle), np.sin(angle)]))
fig, (ax_e, ax_t) = plt.subplots(1, 2, figsize=(9.6, 4.2))
ax_e.plot(cloud[:, 0], cloud[:, 1], ".", color="0.5", ms=3, label="200 relocations")
ax_e.plot(from_grid["x_km"] + ellipse[0], from_grid["y_km"] + ellipse[1], color="tab:red", lw=1.4,
          label="Your 1-sigma error ellipse")
ax_e.plot(from_grid["x_km"] + 2.0 * ellipse[0], from_grid["y_km"] + 2.0 * ellipse[1], color="tab:red", lw=0.9,
          ls="--", label="2-sigma")
ax_e.set_aspect("equal")
ax_e.set_xlabel("East (km)")
ax_e.set_ylabel("North (km)")
ax_e.set_title("(a) Epicenters", fontsize=10, loc="left")
ax_t.plot(cloud[:, 3], cloud[:, 2], ".", color="0.5", ms=3)
ax_t.set_ylim(ax_t.get_ylim()[::-1])
ax_t.set_xlabel("Origin time (s)")
ax_t.set_ylabel("Depth (km)")
ax_t.set_title("(b) Depth against origin time", fontsize=10, loc="left")
fig.legend(loc="upper center", ncol=3, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "practice_monte_carlo.png", dpi=150)

**Explain:** Which unknown is known worst (printed above), and why do its errors go together with the origin-time
errors?

## Part 5 · The Sangju mainshock from your picks (about 40 min, then task 4)

Now the real earthquake: the ML 3.9 Sangju mainshock of 21 July 2019, with your M20 picks and the model kim1983.
Paste `direct_time` and `head_wave` from your M30 notebook into the next cell and run it.

In [ ]:
# Paste your direct_time and head_wave (and, if you like, wadati_fit) of module M30 below this line.

In [ ]:
missing = [name for name in ("direct_time", "head_wave") if not callable(globals().get(name))]
if missing:
    raise NameError(f"{' and '.join(missing)} not found: paste your functions of module M30 into the cell above "
                    "and run it (finish task M30-T2 of module M30 first; task M30-T1 writes direct_time).")
MODEL = st.data.velocity_model(MODEL_NAME)
print(f"{MODEL_NAME}: {MODEL.attrs['citation']}")
print(MODEL.to_string(index=False))
catalog = st.data.kma_catalog()
event = catalog.loc[catalog["event_id"] == KMA_EVENT_ID].iloc[0]
ORIGIN_KMA = UTCDateTime(event["origin_utc"].isoformat())
EV_LAT, EV_LON, KMA_DEPTH_KM = float(event["latitude"]), float(event["longitude"]), float(event["depth_km"])
print(f"\nKMA {KMA_EVENT_ID}: origin {ORIGIN_KMA} (UTC), epicenter {EV_LAT:.2f} N {EV_LON:.2f} E, depth "
      f"{KMA_DEPTH_KM:g} km, ML {event['ml']:.1f}")
print(f"source of the catalog: {catalog.attrs.get('source', 'KMA')}")

**Which wave comes first?** The locator must predict the first arrival, the onset a picker marks. In kim1983, a head
wave along the 15-km boundary overtakes the direct wave beyond the crossover distance. The table gives yours for
sources near the KMA depth.

In [ ]:
cross = []
for depth in (KMA_DEPTH_KM - 4.0, KMA_DEPTH_KM, KMA_DEPTH_KM + 0.9):
    row = {"source depth (km)": depth}
    for n in range(1, len(MODEL)):
        for phase in ("P", "S"):
            label = f"{phase} along {MODEL['top_km'].iloc[n]:g} km" + (" (Moho)" if n == len(MODEL) - 1 else "")
            row[f"{label}: crossover (km)"] = head_wave(np.array([50.0]), depth, MODEL, phase, n)[2]
    cross.append(row)
print("crossover distances of your head_wave (NaN: no head wave from this depth):")
print(pd.DataFrame(cross).round(1).to_string(index=False))

So `tt` is the earliest of your direct wave and your head waves along the crustal boundaries. The helper below
tabulates it every 0.25 km of distance and 0.1 km of depth: a **travel-time table** (a few seconds).

**Watch out:** the lab never locates with Pn and Sn, the head waves along the Moho. Within 100 km they do not come
first here anyway (last two columns).

In [ ]:
def first_arrival_tt(model, direct, head, x_max_km=120.0, dx_km=0.25, z_max_km=35.0, dz_km=0.1):
    """tt(distance_km, depth_km, phase) for the locators: the first arrival in `model` from your M30 functions.

    At every node of a table (distances 0..x_max_km, depths dz_km/2, 3 dz_km/2, ... below z_max_km) the time is
    the earliest of direct(...) and head(..., interface=n)[0] for the boundaries n = 1 .. len(model) - 2 (the Moho,
    the last boundary, is left out: no Pn, no Sn). Between the nodes a cubic spline interpolates; outside the
    table the distance and depth are moved to its edge.
    """
    grid_x = np.arange(0.0, x_max_km + 0.5 * dx_km, dx_km)
    grid_z = np.arange(0.5 * dz_km, z_max_km, dz_km)
    splines = {}
    for phase in ("P", "S"):
        table = np.empty((grid_x.size, grid_z.size))
        for col, depth in enumerate(grid_z):
            earliest = np.asarray(direct(grid_x.copy(), float(depth), model, phase), dtype=float)
            for n in range(1, len(model) - 1):
                crustal = np.asarray(head(grid_x.copy(), float(depth), model, phase, n)[0], dtype=float)
                earliest = np.fmin(earliest, crustal)
            table[:, col] = earliest
        splines[phase] = RectBivariateSpline(grid_x, grid_z, table)

    def tt(distance_km, depth_km, phase):
        where = np.clip(np.asarray(distance_km, dtype=float), grid_x[0], grid_x[-1])
        level = np.clip(np.full(where.shape, float(depth_km)), grid_z[0], grid_z[-1])
        return splines[phase].ev(where, level)

    return tt


start_clock = time.perf_counter()
tt_main = first_arrival_tt(MODEL, direct_time, head_wave)
print(f"travel-time table of {MODEL_NAME} built from your functions in {time.perf_counter() - start_clock:.1f} s")
probe = np.array([5.0, 30.0, 60.0, 90.0])
print("check against your direct_time at", probe.tolist(), "km, source at 10 km (P):",
      np.round(tt_main(probe, 10.0, "P") - direct_time(probe, 10.0, MODEL, "P"), 4).tolist(), "s")

**Your picks in a local frame** centered on the KMA epicenter; times count from the KMA origin time. Picked in
[SeisComP](../../../docs/seiscomp_gui.md)? Its CSV, imported in M20, works here. SeisComP's own location (global model,
fixed depth) is not your result.

In [ ]:
FRAME = Proj(proj="aeqd", lat_0=EV_LAT, lon_0=EV_LON, datum="WGS84")


def to_local(lat, lon):
    """(x_km, y_km) east and north of the KMA epicenter (azimuthal equidistant projection)."""
    east_m, north_m = FRAME(np.asarray(lon, dtype=float), np.asarray(lat, dtype=float))
    return east_m / 1000.0, north_m / 1000.0


def to_geographic(x_km, y_km):
    """(latitude, longitude) in degrees of a point x_km east and y_km north of the KMA epicenter."""
    lon, lat = FRAME(np.asarray(x_km, dtype=float) * 1000.0, np.asarray(y_km, dtype=float) * 1000.0, inverse=True)
    return float(lat), float(lon)


def station_coordinates(sac_dir, names):
    """(latitude, longitude) of each NET.STA in names, from the SAC headers of one event folder (any band)."""
    found = {}
    for path in sorted(Path(sac_dir).glob("*/*.sac")):
        key = ".".join(path.name.split(".")[1:3])
        if key in names and key not in found:
            head = read(str(path), headonly=True)[0].stats.sac
            found[key] = (float(head.stla), float(head.stlo))
    return found


def observation_table(pick_file, sac_dir, origin, max_km):
    """The table the locators read: station, phase, channel, x_km, y_km, t_s (s after origin), time (UTC).

    One row for the first P and one for the first S of each station whose position is in the SAC headers and that
    lies within max_km of the KMA epicenter.
    """
    picks = pd.read_csv(pick_file, dtype=str)
    picks["time"] = [UTCDateTime(t) for t in picks["time_utc"]]
    picks = picks.sort_values("time").drop_duplicates(["station", "phase"])
    where = station_coordinates(sac_dir, set(picks["station"]))
    rows = []
    for _, pick in picks.iterrows():
        if pick["station"] in where and pick["phase"] in ("P", "S"):
            east, north = to_local(*where[pick["station"]])
            if np.hypot(east, north) <= max_km:
                rows.append({"station": pick["station"], "phase": pick["phase"], "channel": pick["channel"],
                             "x_km": float(east), "y_km": float(north), "t_s": pick["time"] - origin,
                             "time": pick["time"]})
    table = pd.DataFrame(rows)
    table["epi_km"] = np.hypot(table["x_km"], table["y_km"])
    return table.sort_values(["epi_km", "phase"]).reset_index(drop=True)


SAC_DIR = st.data.root() / DATASET / "stp_download" / "SAC" / ORIGIN_KMA.strftime("%Y%m%d%H%M%S")
if not SAC_DIR.is_dir():
    raise FileNotFoundError(f"There are no waveforms of this event in {SAC_DIR}. Fetch the {DATASET} data with STP "
                            "first (module M03), or set DATASET in the parameters cell to the folder you used.")
if not Path(PICKS_CSV).is_file():
    raise FileNotFoundError(f"{PICKS_CSV} is missing. This module locates the mainshock with your own picks: "
                            "finish task M20-T5 of module M20 first (its picker saves the file).")
obs_main = observation_table(PICKS_CSV, SAC_DIR, ORIGIN_KMA, MAX_DISTANCE_KM)
print(f"{len(obs_main)} picks ({(obs_main['phase'] == 'P').sum()} P, {(obs_main['phase'] == 'S').sum()} S) at "
      f"{obs_main['station'].nunique()} stations within {MAX_DISTANCE_KM:g} km of the KMA epicenter")
print(obs_main.drop(columns=["time"]).round(3).to_string(index=False))
GRID_X = np.arange(-GRID_HALF_KM, GRID_HALF_KM + 0.5 * GRID_STEP_KM, GRID_STEP_KM)
GRID_Y = GRID_X.copy()
GRID_Z = np.arange(GRID_DEPTHS_KM[0], GRID_DEPTHS_KM[1] + 0.5 * GRID_DEPTHS_KM[2], GRID_DEPTHS_KM[2])

### M31-T4 · The Sangju mainshock and the KMA catalog · core · about 60 min

Locate the mainshock with your picks `obs_main`, your table `tt_main` and your three functions: `grid_search` over
`GRID_X`, `GRID_Y`, `GRID_Z`; `geiger` from its best node; `location_errors` with
$\hat\sigma = \mathrm{RMS}\sqrt{N/(N-4)}$, the pick error estimated from your $N$ residuals. Keep the results as
`fit_main` and `errors_main`, then fill in the dict `sangju`; the code cell lists its keys.

Example of `origin_utc` (an invented time): `"2031-04-05T06:07:08.090000Z"`. Always in UTC.

In [ ]:
# Before you code: KMA rounds the epicenter to 0.01 degree (about 1 km). How far from KMA's hypocenter do
# you expect yours?
st.predict("M31-T4", """

""")

In [ ]:
# Locate the mainshock: grid search, then geiger from its best node, then location_errors with sigma from your
# residuals. Keep fit_main and errors_main, then fill in sangju with these keys:
#   x_km, y_km, depth_km : your hypocenter in the local frame (km)
#   t0_s                 : your origin time in seconds after the KMA origin time ORIGIN_KMA
#   origin_utc           : the same time as ISO 8601 text in UTC
#   latitude, longitude  : your epicenter in degrees (to_geographic converts x_km, y_km)
#   rms_s                : your RMS residual (s)
#   se                   : a list of your four standard errors (x, y, z in km; origin time in s)
#   n_picks              : the number of picks you used (every row of obs_main)
# YOUR CODE HERE
raise NotImplementedError("M31-T4: write your code here")

In [ ]:
st.check("M31-T4", answer=sangju, obs=obs_main, tt=tt_main)

Your location next to the KMA catalog, and your residuals station by station.

**Watch out:** model errors are not random from pick to pick. So standard errors from real picks are a lower limit
of the true error.

In [ ]:
sigma_used = fit_main["rms_s"] * np.sqrt(len(obs_main) / (len(obs_main) - 4))
print(f"your location: {sangju['latitude']:.4f} N {sangju['longitude']:.4f} E, depth {sangju['depth_km']:.2f} "
      f"km, origin {sangju['origin_utc']} ({fit_main['n_iter']} steps, converged {fit_main['converged']})")
print(f"RMS {fit_main['rms_s']:.3f} s from {len(obs_main)} picks; sigma {sigma_used:.3f} s; standard errors x "
      "{:.2f}, y {:.2f}, z {:.2f} km, origin time {:.3f} s; depth-origin time correlation {:.2f}".format(
          *errors_main["se"], errors_main["corr_zt"]))
print(f"KMA catalog:   {EV_LAT:.4f} N {EV_LON:.4f} E, depth {KMA_DEPTH_KM:.2f} km, origin {ORIGIN_KMA}")
print(f"you minus KMA: epicenter {np.hypot(sangju['x_km'], sangju['y_km']):.2f} km away (east "
      f"{sangju['x_km']:+.2f}, north {sangju['y_km']:+.2f} km), depth {sangju['depth_km'] - KMA_DEPTH_KM:+.2f} km, "
      f"origin time {sangju['t0_s']:+.2f} s")
angles_deg = np.degrees(np.arctan2(obs_main["x_km"] - sangju["x_km"], obs_main["y_km"] - sangju["y_km"]))
azimuths = np.sort(np.mod(angles_deg, 360.0))
print(f"largest azimuthal gap of your stations: {np.max(np.diff(np.append(azimuths, azimuths[0] + 360.0))):.0f} "
      "degrees (glossary azimuthal_gap)")
resid_main = obs_main[["station", "phase", "epi_km"]].copy()
resid_main["residual_s"] = fit_main["residual_s"]
for phase in ("P", "S"):
    part = resid_main[resid_main["phase"] == phase]["residual_s"]
    print(f"{phase}: {len(part)} residuals, mean {part.mean():+.3f} s, smallest {part.min():+.3f}, largest "
          f"{part.max():+.3f} s")
print(resid_main.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(6.4, 3.8))
for phase, color in (("P", "tab:blue"), ("S", "tab:red")):
    part = resid_main[resid_main["phase"] == phase]
    ax.plot(part["epi_km"], part["residual_s"], "o", color=color, ms=6, label=f"{phase} residuals")
ax.axhline(0.0, color="0.5", lw=0.8)
ax.set_xlabel("Epicentral distance from the KMA epicenter (km)")
ax.set_ylabel("Observed - predicted (s)")
ax.set_title(f"Your residuals at your location, {MODEL_NAME}", fontsize=10)
fig.legend(loc="upper center", ncol=2, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.88))
fig.savefig(Path(FIGURE_DIR) / "mainshock_residuals.png", dpi=150)

**Explain:** What pattern do your P and S residuals show (printed above), and what does it say about kim1983?

**Optional: explorer E5** shows the RMS of your `grid_search` in map view and in a depth section. Leave out the
nearest station, then the two nearest, and watch the depth section.

In [ ]:
# Optional explorer E5 (not checked): misfit slices of your grid_search; leave stations out and watch.
st.widgets.misfit_explorer(obs_main, tt_main, grid_search=grid_search, wadati=globals().get("wadati_fit"),
                           marks={"KMA hypocenter": (0.0, 0.0, KMA_DEPTH_KM)}, half_km=GRID_HALF_KM,
                           step_km=GRID_STEP_KM, depths=GRID_Z, depth_km=sangju["depth_km"])

## Part 6 · A small earthquake in 2018 (about 40 min of picking, then task 5)

Seventeen months before the mainshock, an ML 1.4 earthquake occurred in the same area. Only nearby stations record
a small earthquake well, so it has few picks. You pick it, locate it, and test what the nearest station adds.

In [ ]:
small = catalog.loc[catalog["event_id"] == SMALL_EVENT_ID].iloc[0]
ORIGIN_SMALL = UTCDateTime(small["origin_utc"].isoformat())
SMALL_LAT, SMALL_LON = float(small["latitude"]), float(small["longitude"])
print(f"KMA {SMALL_EVENT_ID}: origin {ORIGIN_SMALL} (UTC), epicenter {SMALL_LAT:.2f} N {SMALL_LON:.2f} E, depth "
      f"{float(small['depth_km']):g} km, ML {small['ml']:.1f}")
SAC_DIR_SMALL = st.data.root() / DATASET / "stp_download" / "SAC" / ORIGIN_SMALL.strftime("%Y%m%d%H%M%S")
if not (SAC_DIR_SMALL / SMALL_BAND).is_dir():
    raise FileNotFoundError(f"There are no {SMALL_BAND} waveforms of this event in {SAC_DIR_SMALL}. Fetch the "
                            f"{DATASET} data with STP first (module M03).")

**Pick it** with the picker E2 of M20 (about 40 min): P on the vertical, S on a horizontal channel, with
uncertainties. Pick at least seven stations, the nearest among them: the depth scans drop the two nearest and still
need five. Skip a station whose onset you cannot see. Press Save.

**Watch out:** keep the causal 2-15 Hz filter. A zero-phase filter makes picks early (M10).

In [ ]:
# The picker E2: one station at a time; your picks go to SMALL_PICKS_CSV when you press Save.
picker_small = st.widgets.phase_picker(SAC_DIR_SMALL / SMALL_BAND, origin=ORIGIN_SMALL,
                                       epicenter=(SMALL_LAT, SMALL_LON), max_km=SMALL_MAX_KM,
                                       csv_path=SMALL_PICKS_CSV, window=(-2.0, 25.0), fmin=2.0, fmax=15.0)
picker_small

Your 2018 picks, in the same frame and with the same travel-time table (times from KMA's 2018 origin time).

In [ ]:
if not Path(SMALL_PICKS_CSV).is_file():
    raise FileNotFoundError(f"{SMALL_PICKS_CSV} is missing: pick the 2018 earthquake in the picker above and press "
                            "Save (or call picker_small.save()).")
obs_small = observation_table(SMALL_PICKS_CSV, SAC_DIR_SMALL, ORIGIN_SMALL, MAX_DISTANCE_KM)
NEAREST = obs_small.sort_values("epi_km")["station"].drop_duplicates().tolist()
print(f"{len(obs_small)} picks at {len(NEAREST)} stations; nearest first: {NEAREST}")
print(obs_small.drop(columns=["time"]).round(3).to_string(index=False))

### M31-T5 · A depth scan · core · about 75 min

Write `depth_scan(obs, tt, depths, start)`. At each fixed depth, it finds the best epicenter and origin time with
your Geiger step (three unknowns), and the RMS there. RMS against depth is a **depth scan**; the docstring gives the
details.

Example: for exact picks from a source 8 km deep, the row at 8 km has RMS 0.

Run the colleague's cell below first. Not allowed: the libraries of task 2.

In [ ]:
# A colleague located the 2018 earthquake with your grid_search and geiger twice: with all your picks, and without
# the station nearest to the epicenter. Run it as it is.
colleague = {}
without_nearest = obs_small[obs_small["station"] != NEAREST[0]]
for label, table in (("all stations", obs_small), (f"without {NEAREST[0]}", without_nearest)):
    table = table.reset_index(drop=True)
    first_guess = grid_search(table, tt_main, GRID_X, GRID_Y, GRID_Z)
    colleague[label] = geiger(table, tt_main, (first_guess["x_km"], first_guess["y_km"], first_guess["z_km"],
                                               first_guess["t0_s"]))
    print(f"colleague, {label}: depth {colleague[label]['z_km']:.1f} km, RMS {colleague[label]['rms_s']:.3f} s, "
          f"{len(table)} picks")
print("colleague's note: both fits have about the same RMS, so the nearest station can be left out without loss; "
      "the depth of the 2018 earthquake is known either way.")

In [ ]:
# Before you code: do you agree with the colleague's note? Think of how the RMS will change with depth.
st.predict("M31-T5", """

""")

In [ ]:
def depth_scan(obs, tt, depths, start):
    """Best epicenter and origin time, and the RMS, at each of a list of fixed depths.

    obs    : table with phase, x_km, y_km, t_s (as in task 1); do not change it
    tt     : function tt(distance_km, depth_km, phase) -> travel times (s)
    depths : 1-D array of fixed depths (km)
    start  : (x_km, y_km, t0_s) for the first depth
    At each depth, in the order given: Geiger's step on x, y and t0 only (the columns dT/dx and dT/dy by central
    differences with h = 0.01 km, and ones; no damping), with the depth held at that value, starting from the
    answer at the depth before (the first depth from start). Stop when a step moves the epicenter by less than
    0.001 km, or after 30 steps.
    returns: pandas DataFrame with the columns depth_km, x_km, y_km, t0_s, rms_s (one row per depth, in the order
             given; rms_s after the last step)
    """
    # Step 1: the predicted times of a trial (x, y) at a fixed depth (as in task 2).
    # Step 2: at one depth, Geiger's step with three columns (x, y, ones): solve G^T G dm = G^T r, update
    #         x, y, t0, repeat until the epicenter moves less than 0.001 km (at most 30 steps).
    # Step 3: start each depth from the answer of the one before; record the RMS at the end.
    # YOUR CODE HERE
    raise NotImplementedError("M31-T5: write your code here")

In [ ]:
st.check("M31-T5", func=depth_scan)

Your depth scans: with all stations, without the nearest one, and without the two nearest. A depth is within one
standard error of the best one when $N(\mathrm{RMS}^2 - \mathrm{RMS}_{\min}^2)/\hat\sigma^2 \le 1$; the cell prints
that range for each set.

In [ ]:
SCAN = np.arange(SCAN_DEPTHS_KM[0], SCAN_DEPTHS_KM[1] + 0.5 * SCAN_DEPTHS_KM[2], SCAN_DEPTHS_KM[2])
sets = {"all stations": obs_small,
        f"without {NEAREST[0]}": obs_small[obs_small["station"] != NEAREST[0]],
        f"without {NEAREST[0]} and {NEAREST[1]}": obs_small[~obs_small["station"].isin(NEAREST[:2])]}
scans, fits_small, summary = {}, {}, []
for label, table in sets.items():
    table = table.reset_index(drop=True)
    n_stations = table["station"].nunique()
    if n_stations < 5:                           # 4 unknowns: sigma needs more picks than that, and a scan needs more
        raise ValueError(f"The set '{label}' has picks at {n_stations} stations ({len(table)} picks); every set needs "
                         "at least 5 stations. Pick at least 7 stations of the 2018 earthquake in the picker above "
                         "and press Save.")
    guess = grid_search(table, tt_main, GRID_X, GRID_Y, GRID_Z)
    fit = geiger(table, tt_main, (guess["x_km"], guess["y_km"], guess["z_km"], guess["t0_s"]))
    n = len(table)
    sigma = fit["rms_s"] * np.sqrt(n / (n - 4))
    try:
        err = location_errors(fit["G"], sigma)
        se_depth, corr_zt = err["se"][2], err["corr_zt"]
    except np.linalg.LinAlgError:                # G^T G cannot be inverted: some direction is not fixed at all
        err, se_depth, corr_zt = None, np.inf, np.nan
    scan = depth_scan(table, tt_main, SCAN, (fit["x_km"], fit["y_km"], fit["t0_s"]))
    scans[label], fits_small[label] = scan, (fit, err)
    best = scan["rms_s"].min()
    inside = scan["depth_km"][n * (scan["rms_s"] ** 2 - best ** 2) / sigma ** 2 <= 1.0]
    around = np.abs(scan["depth_km"] - scan.loc[scan["rms_s"].idxmin(), "depth_km"]) <= 3.0
    slope = np.polyfit(scan["depth_km"][around], scan["t0_s"][around], 1)[0]    # within 3 km of the best depth
    summary.append({"picks": label, "N": n, "depth (km)": fit["z_km"], "RMS (s)": fit["rms_s"],
                    "sigma (s)": sigma, "se depth (km)": se_depth, "corr depth-t0": corr_zt,
                    "scan: best depth (km)": scan.loc[scan["rms_s"].idxmin(), "depth_km"],
                    "scan: depths within 1 se (km)": f"{inside.min():g}-{inside.max():g}",
                    "scan: t0 change per km near the best (s)": slope})
print(pd.DataFrame(summary).round(3).to_string(index=False))
for label, scan in scans.items():                # every depth where the RMS is lower than at both neighbours
    r = scan["rms_s"].to_numpy()
    lower_left = np.r_[True, r[1:] < r[:-1]]
    lower_right = np.r_[r[:-1] <= r[1:], True]
    minima = scan["depth_km"][lower_left & lower_right]
    print(f"{label}: local minima of the RMS at", ", ".join(f"{d:g} km" for d in minima))
print(f"KMA catalog depth of this earthquake: {float(small['depth_km']):g} km; your mainshock depth: "
      f"{sangju['depth_km']:.2f} km")

fig, (ax_r, ax_t) = plt.subplots(1, 2, figsize=(10.0, 4.0))
for (label, scan), color in zip(scans.items(), ("k", "tab:orange", "tab:purple")):
    ax_r.plot(scan["depth_km"], scan["rms_s"], color=color, lw=1.5, label=label)
    ax_t.plot(scan["depth_km"], scan["t0_s"], color=color, lw=1.5)
ax_r.axvline(float(small["depth_km"]), color="0.5", ls=":", lw=1.0, label="KMA catalog depth")
ax_t.axvline(float(small["depth_km"]), color="0.5", ls=":", lw=1.0)
ax_r.set_xlabel("Fixed depth (km)")
ax_r.set_ylabel("RMS residual (s)")
ax_r.set_title("(a) Misfit against depth", fontsize=10, loc="left")
ax_t.set_xlabel("Fixed depth (km)")
ax_t.set_ylabel("Best origin time after KMA's (s)")
ax_t.set_title("(b) Origin time against depth", fontsize=10, loc="left")
fig.legend(loc="upper center", ncol=4, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.88))
fig.savefig(Path(FIGURE_DIR) / "depth_scan_2018.png", dpi=150)

**Explain:** Is the colleague's note right? Answer with the depths within one standard error, with and without the
nearest station (printed above).

**The two earthquakes on a map:** (a) your stations, (b) your two epicenters with their 1-sigma error ellipses, and
KMA's.

In [ ]:
import pygmt

fit_small, err_small = fits_small["all stations"]
lat_small, lon_small = to_geographic(fit_small["x_km"], fit_small["y_km"])
sta_where = station_coordinates(SAC_DIR, set(obs_main["station"]))
sta_where.update(station_coordinates(SAC_DIR_SMALL, set(obs_small["station"])))
sta_lat = np.array([v[0] for v in sta_where.values()])
sta_lon = np.array([v[1] for v in sta_where.values()])


def ellipse_row(lon, lat, cov):
    """[lon, lat, azimuth (deg from north), major, minor (km, full axes)] of the 1-sigma ellipse of cov[:2, :2]."""
    values, vectors = np.linalg.eigh(np.asarray(cov)[:2, :2])
    major = vectors[:, 1]
    azimuth = float(np.degrees(np.arctan2(major[0], major[1])) % 180.0)
    return [lon, lat, azimuth, 2.0 * np.sqrt(values[1]), 2.0 * np.sqrt(values[0])]


region_a = [float(sta_lon.min()) - 0.1, float(sta_lon.max()) + 0.1, float(sta_lat.min()) - 0.08,
            float(sta_lat.max()) + 0.08]
region_b = [EV_LON - 0.06, EV_LON + 0.06, EV_LAT - 0.045, EV_LAT + 0.045]
with st.plot.gmt_style(FONT_TITLE="11p,Helvetica"):
    gmt_fig = pygmt.Figure()
    gmt_fig.coast(region=region_a, projection="M8c", land="gray95", water="white", shorelines="0.25p,gray50")
    gmt_fig.plot(x=sta_lon, y=sta_lat, style="t0.25c", fill="royalblue", pen="0.4p,black", label="Stations used")
    gmt_fig.plot(x=[region_b[0], region_b[1], region_b[1], region_b[0], region_b[0]],
                 y=[region_b[2], region_b[2], region_b[3], region_b[3], region_b[2]], pen="0.8p,red",
                 label="Box of panel b")
    gmt_fig.legend(position="jBL+o0.15c", box="+gwhite+p0.4p")
    gmt_fig.basemap(frame=["af", "+t(a) Stations"], map_scale=st.plot.scale_bar_spec(region_a, corner="BR"))
    gmt_fig.shift_origin(xshift="9.2c")
    gmt_fig.coast(region=region_b, projection="M8c", land="gray95", water="white", shorelines="0.25p,gray50")
    gmt_fig.plot(data=[ellipse_row(sangju["longitude"], sangju["latitude"], errors_main["cov"])], style="E",
                 pen="1p,red3")
    gmt_fig.plot(x=[sangju["longitude"]], y=[sangju["latitude"]], style="a0.35c", fill="red3", pen="0.4p,black",
                 label="Your mainshock 2019")
    gmt_fig.plot(data=[ellipse_row(lon_small, lat_small, err_small["cov"])], style="E", pen="1p,darkorange")
    gmt_fig.plot(x=[lon_small], y=[lat_small], style="a0.3c", fill="darkorange", pen="0.4p,black",
                 label="Your 2018 earthquake")
    gmt_fig.plot(x=[EV_LON], y=[EV_LAT], style="x0.3c", pen="1.5p,red3", label="KMA 2019")
    gmt_fig.plot(x=[SMALL_LON], y=[SMALL_LAT], style="x0.3c", pen="1.5p,darkorange", label="KMA 2018")
    gmt_fig.legend(position="jTL+o0.15c", box="+gwhite+p0.4p")
    gmt_fig.basemap(frame=["af", "+t(b) Epicenters"], map_scale=st.plot.scale_bar_spec(region_b, corner="BR"))
gmt_fig.savefig(Path(FIGURE_DIR) / "sangju_epicenters_map.png", dpi=200)
gmt_fig.show()

## Part 7 · HYPOINVERSE, a production locator (about 20 min, then task 6)

**HYPOINVERSE** is the U.S. Geological Survey's locator, and the one PocketQuake runs: Geiger's method in a layered
model. It adds **distance weighting** (far stations count less) and **residual weighting** (picks with large
residuals count less). Each location starts at a **trial depth** (`ZTR`). The helpers below run it with the lab's
settings. The cell prints PocketQuake's commands next to the ones sent here: the same, line by line, except the file
names.

In [ ]:
HYP_COMMANDS = """REP T T
CON 50
MIN 4
ZTR {trial:g} F
DIS 4 50 1 3
RMS 4 0.12 2 4
ERF T
TOP F
LST 2 0 1
KPR 3
H71 4 1 3
LET 5 2 0 0 0
STA 'stations.sta'
CRH 1 'model_p.crh'
CRH 2 'model_s.crh'
SAL 1 2
PHS 'event.phs'
FIL
PRT 'run.prt'
SUM 'run.sum'
ARC 'run.arc'
LOC
STO
"""


def hypoinverse_program():
    """The path of hyp1.40, which bin/sgtl-install tools put in ~/sgtl/tools/bin."""
    search = os.pathsep.join(p for p in (st.pq.tools_env().get("TOOLS_PATH", ""), os.environ.get("PATH", "")) if p)
    found = shutil.which("hyp1.40", path=search)
    if found is None:
        raise FileNotFoundError("HYPOINVERSE (hyp1.40) is not installed. In a terminal: bin/sgtl-install tools "
                                "(read the dry run), then bin/sgtl-install tools --apply")
    return found


def deg_min(value, positive, negative):
    """Whole degrees, decimal minutes and the hemisphere letter of a latitude or longitude."""
    whole = int(abs(value))
    return whole, (abs(value) - whole) * 60.0, positive if value >= 0 else negative


def write_hypoinverse_files(folder, picks, coords, model, origin, lat, lon, phase_lines):
    """Station file, P and S model files and the phase file (header, your lines, closing line) in folder."""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    with open(folder / "stations.sta", "w") as out:
        for name in sorted(set(picks["station"])):
            network, code = name.split(".")
            la_d, la_m, ns = deg_min(coords[name][0], "N", "S")
            lo_d, lo_m, ew = deg_min(coords[name][1], "E", "W")
            out.write(f"{code:<5} {network:<2}  HHZ  {la_d:2d} {la_m:7.4f}{ns}{lo_d:3d} {lo_m:7.4f}{ew}   0\n")
    for suffix, column in (("p", "vp"), ("s", "vs")):
        layers = "".join(f" {v:.2f} {top:.2f}\n" for v, top in zip(model[column], model["top_km"]))
        (folder / f"model_{suffix}.crh").write_text(f"{MODEL_NAME} {suffix.upper()}\n" + layers)
    la_d, la_m, ns = deg_min(lat, "N", " ")
    lo_d, lo_m, ew = deg_min(lon, "E", "W")
    header = (origin.strftime("%Y%m%d%H%M%S") + f"{int(round(origin.microsecond / 1e4)):02d}"
              + f"{la_d:2d}{ns}{int(la_m * 100):4d}{lo_d:3d}{ew}{int(lo_m * 100):4d}")
    closing = " " * 66 + "1"
    (folder / "event.phs").write_text("\n".join([header, *phase_lines, closing]) + "\n")


def run_hypoinverse(folder, trial_depth_km):
    """Run HYPOINVERSE in folder with the lab's settings and the given trial depth; return its summary line."""
    done = subprocess.run([hypoinverse_program()], input=HYP_COMMANDS.format(trial=trial_depth_km), cwd=folder,
                          capture_output=True, text=True, timeout=300)
    if done.returncode != 0 or not (Path(folder) / "run.sum").is_file():
        raise RuntimeError(f"HYPOINVERSE failed in {folder}: {done.stdout[-800:]}")
    return read_hypoinverse_summary(Path(folder) / "run.sum")


def read_hypoinverse_summary(path):
    """The first earthquake of a HYPOINVERSE summary file (H71 4 1 3, comma-separated) as a dict."""
    table = pd.read_csv(path, skipinitialspace=True)
    table.columns = [str(c).strip() for c in table.columns]
    line = table.iloc[0]
    minute = str(line[table.columns[0]]).strip().replace("/", "-").replace(" ", "T")
    return {"latitude": float(line["LAT"]), "longitude": float(line["LON"]), "depth_km": float(line["DEPTH"]),
            "origin": UTCDateTime(minute + ":00") + float(line["SEC"]), "rms_s": float(line["RMS"]),
            "erh_km": float(line["ERH"]), "erz_km": float(line["ERZ"]), "phases": int(line["NUM"]),
            "gap_deg": int(line["GAP"])}


print("HYPOINVERSE:", hypoinverse_program())

# PocketQuake's location commands (left, from your copy; the values in braces are printed below them) next to the
# commands the helpers send (right, with the trial depth TRIAL_DEPTH_KM).
PQ_PIPELINE = st.pq.pipeline_dir() / "pipeline"
if (PQ_PIPELINE / "core" / "hypoinverse.py").is_file():
    template = (PQ_PIPELINE / "core" / "hypoinverse.py").read_text(encoding="utf-8")
    lab_lines = template.split('HYP_TEMPLATE = """', 1)[1].split('"""', 1)[0].strip().splitlines()
    sent_lines = HYP_COMMANDS.format(trial=TRIAL_DEPTH_KM).strip().splitlines()
    print(f"\n{'PocketQuake (HYP_TEMPLATE)':<36}this module (HYP_COMMANDS)")
    for lab_line, sent_line in zip(lab_lines, sent_lines):
        print(f"{lab_line:<36}{sent_line}")
    print(f"lines: {len(lab_lines)} and {len(sent_lines)}")
    for source, word, count in ((PQ_PIPELINE / "clusters" / "_base.py", "STD_HYP =", 2),
                                (PQ_PIPELINE / "config.py", "LET: tuple", 1)):
        lines = source.read_text(encoding="utf-8").splitlines()
        at = next(k for k, line in enumerate(lines) if word in line)
        print("\n".join(f"{source.name}:{at + 1 + j}: {lines[at + j].strip()}" for j in range(count)))
else:
    print("PocketQuake is not installed in", st.pq.clone_dir(), "(module M21 installs it), so its template cannot be "
          "shown here.")

HYPOINVERSE reads your picks from a **phase file** in a **fixed-width format**: every value has its own columns.
Numbers carry an **implied decimal point** (8.25 s is written `825`). Each pick has a **weight code** from 0 (full
weight) to 4 (none). One column off, and a value is read as something else.

### M31-T6 · Phase lines for HYPOINVERSE · core · about 60 min

Write `hypoinverse_lines(picks)`: one line of the phase file per pick. The docstring gives the columns. The cells
after the check then run HYPOINVERSE on your mainshock picks three times.

Example (invented): P on `HHZ` of `XX.AB1` at `2031-04-05T06:07:08.254Z`, weight 0, gives

```text
AB1  XX  HHZ IP 0203104050607  825
```

Not allowed: PocketQuake's code (`pipeline`, `pocketquake`), `obspy.io` or the course package.

In [ ]:
# Before you code: with the same picks and model, how close to your geiger location will HYPOINVERSE put
# the mainshock?
st.predict("M31-T6", """

""")

In [ ]:
def hypoinverse_lines(picks):
    """HYPOINVERSE phase-file lines (Y2000 format), one per pick.

    picks   : table with station (NET.STA: a network code of 2 characters, a station code of 1 to 5), phase ("P" or
              "S"), channel (3 characters), time (obspy UTCDateTime, in UTC) and weight (int 0-4); any order; do
              not change it
    returns : list of str, one line per row of picks, in its order, without line breaks
    Round each time to the nearest 0.01 s first, then take its minute and seconds. Columns, counted from 1;
    every other column up to the last one used is blank:
        columns  P line                                       S line
        1-5      station code, left-justified                 the same
        6-7      network code                                 the same
        10-12    channel                                      the same
        14-15    IP                                           blank
        17       weight digit                                 blank
        18-29    YYYYMMDDHHMM of the pick                     the same
        30-34    seconds after that minute in hundredths,     blank
                 right-justified (8.25 s: '  825')
        42-46    blank                                        seconds in hundredths, right-justified
        47-48    blank                                        ES
        50       blank                                        weight digit
    """
    # Step 1: for each pick, split NET.STA; round the time to 0.01 s; its minute as YYYYMMDDHHMM and the seconds
    #         after that minute in hundredths (an integer).
    # Step 2: build the line with f-strings: :<5 pads on the right, :5d right-justifies an integer in 5 columns.
    # Step 3: P and S put the phase, the weight and the seconds in different columns (see the docstring).
    # YOUR CODE HERE
    raise NotImplementedError("M31-T6: write your code here")

In [ ]:
st.check("M31-T6", func=hypoinverse_lines)

The production runs on your mainshock picks (all weights 0): run 1 with the lab's settings, run 2 the same again,
run 3 with `CHANGED_TRIAL_DEPTH_KM`. Then the same change without the nearest station. Is the rerun identical, file by
file?

In [ ]:
HYP_DIR = Path(RESULT_DIR) / "hypoinverse"
coords_main = station_coordinates(SAC_DIR, set(obs_main["station"]))
hyp_picks = obs_main[["station", "phase", "channel", "time"]].assign(weight=0)
print("your first three phase lines:", *hypoinverse_lines(hyp_picks)[:3], sep="\n")
runs = {}
for label, trial in (("run 1", TRIAL_DEPTH_KM), ("run 2 (rerun)", TRIAL_DEPTH_KM),
                     (f"run 3 (trial depth {CHANGED_TRIAL_DEPTH_KM:g} km)", CHANGED_TRIAL_DEPTH_KM)):
    folder = HYP_DIR / label.split(" (")[0].replace(" ", "")
    write_hypoinverse_files(folder, hyp_picks, coords_main, MODEL, ORIGIN_KMA, EV_LAT, EV_LON,
                            hypoinverse_lines(hyp_picks))
    runs[label] = run_hypoinverse(folder, trial)
    time.sleep(1.1)                      # one run per second at most, as when you run the cells one by one
for name in ("run.sum", "run.arc", "run.prt"):
    first_lines = (HYP_DIR / "run1" / name).read_text().splitlines()
    rerun_lines = (HYP_DIR / "run2" / name).read_text().splitlines()
    differ = [(a, b) for a, b in zip(first_lines, rerun_lines) if a != b]
    same = filecmp.cmp(HYP_DIR / "run1" / name, HYP_DIR / "run2" / name, shallow=False)
    print(f"run 2 {name} identical to run 1: {same}; lines that differ: {len(differ)}"
          + (f" (run 1: {differ[0][0].strip()!r}; run 2: {differ[0][1].strip()!r})" if differ else ""))

nearest_main = obs_main.sort_values("epi_km")["station"].iloc[0]
fewer = obs_main[obs_main["station"] != nearest_main].reset_index(drop=True)
for trial in (TRIAL_DEPTH_KM, CHANGED_TRIAL_DEPTH_KM):
    folder = HYP_DIR / f"without_nearest_trial{trial:g}"
    picks_fewer = fewer[["station", "phase", "channel", "time"]].assign(weight=0)
    write_hypoinverse_files(folder, picks_fewer, coords_main, MODEL, ORIGIN_KMA, EV_LAT, EV_LON,
                            hypoinverse_lines(picks_fewer))
    runs[f"without {nearest_main}, trial depth {trial:g} km"] = run_hypoinverse(folder, trial)
guess_fewer = grid_search(fewer, tt_main, GRID_X, GRID_Y, GRID_Z)
mine_fewer = geiger(fewer, tt_main, (guess_fewer["x_km"], guess_fewer["y_km"], guess_fewer["z_km"],
                                     guess_fewer["t0_s"]))

lat_fewer, lon_fewer = to_geographic(mine_fewer["x_km"], mine_fewer["y_km"])
located = {"your geiger": {"latitude": sangju["latitude"], "longitude": sangju["longitude"],
                           "depth_km": sangju["depth_km"], "origin": ORIGIN_KMA + sangju["t0_s"],
                           "rms_s": sangju["rms_s"]}}
located.update(list(runs.items())[:3])
located[f"your geiger without {nearest_main}"] = {"latitude": lat_fewer, "longitude": lon_fewer,
                                                  "depth_km": mine_fewer["z_km"],
                                                  "origin": ORIGIN_KMA + mine_fewer["t0_s"],
                                                  "rms_s": mine_fewer["rms_s"]}
located.update(list(runs.items())[3:])
compare = []
for label, loc in located.items():
    east, north = to_local(loc["latitude"], loc["longitude"])
    compare.append({"location": label, "latitude": loc["latitude"], "longitude": loc["longitude"],
                    "east (km)": float(east), "north (km)": float(north), "depth (km)": loc["depth_km"],
                    "origin - KMA (s)": loc["origin"] - ORIGIN_KMA, "RMS (s)": loc["rms_s"],
                    "ERH (km)": loc.get("erh_km", np.nan), "ERZ (km)": loc.get("erz_km", np.nan)})
compare = pd.DataFrame(compare)
print(compare.round(4).to_string(index=False))
compare.to_csv(Path(RESULT_DIR) / "M31_hypoinverse_runs.csv", index=False)

**Explain:** What did the changed trial depth do, with all stations and without the nearest one, and why?

## Wrap-up

Your progress in this module (from `work/.sgtl/progress.jsonl`):

In [ ]:
st.progress("M31")

**Optional:** locate both earthquakes with `kim2011` (set `MODEL_NAME`, rerun from Part 5). Does the P-S split of
your residuals change?

**Read more:** Havskov et al. (2012), the location problem; Husen and Hardebeck (2010), location errors; Klein
(2002), the HYPOINVERSE manual.

Next: M32 runs HYPOINVERSE inside PocketQuake on six Sangju earthquakes.